# 1. Adresování a správa paměti - Garbage collecting, Reference/ukazatele, Struktura paměti programu

### Struktura paměti programu
* Když operační systém spustí program, vyhradí mu v RAM virtuální adresní prostor, který se logicky dělí na několik částí:
    * Kódový segment - Zde jsou uloženy přeložené strojové instrukce programu, je obvykle určena pouze pro čtení
    * Datový segment - Pro uložení globálních a statických proměnných, které existují po celou dobu běhu programu
    * Zásobník - Paměť s architekturou LIFO, ukládají se sem lokální proměnné uvnitř funkcí a návratové adresy při zanořování do dalších funkcí, velmi rychlý, jeho velikost omezena, jakmile funkce skončí, její proměnné se ze zásobníku automaticky vymažou
    * Halda - Prostor určený pro dynamickou alokaci paměti za běhu programu, v objektových jazycích (jako je Python nebo Java) se zde vytvářejí všechny instance tříd, pole a objekty, paměť na haldě se nepromazává automaticky koncem bloku, musí se aktivně spravovat

### Ukazatele vs. Reference
* Ukazatel (C/C++) - Proměnná, jejíž hodnotou je přímá fyzická adresa jiného místa v paměti, programátor s ním může dělat matematiku a musí dávat pozor, aby neukázal do paměti cizího programu
* Reference (Python, C#, Java) - Jedná se o bezpečnější abstrakci nad ukazatelem, reference ukazuje na objekt ležící na haldě, ale programátor nemá přístup k její skutečné paměťové adrese a nemůže dělat paměťovou aritmetiku, v Pythonu neexistují klasické proměnné jako „krabice na hodnoty“, existují pouze reference, které se lepí na objekty v haldě

### Správa paměti a Garbage Collecting
* V C nebo C++ musí vývojář o paměť na haldě ručně žádat (`malloc` / `new`) a ručně vracet (`free` / `delete`), pokud zapomene, vzniká Memory Leak
* Moderní jazyky používají automatickou správu, v Pythonu funguje primárně na Reference Counting, každý objekt si pamatuje, kolik referencí na něj ukazuje, jakmile toto číslo klesne na 0, objekt se smaže
* Garbage Collector - Počítání referencí má slabinu – Cyklické reference, pokud se tyto objekty odpojí od zbytku programu, jejich počítadlo neklesne pod 1, proto na pozadí běží ještě Garbage Collector, občas zastaví program, prohledá paměť, najde tyto oddělené kusy a z paměti je uvolní

**1. Reference v paměti**

In [5]:
from asyncio import timeout
from time import sleep

seznam_a = [1, 2, 3]
seznam_b = seznam_a  #novy odkaz na stejne misto

print(f"Adresa a: {id(seznam_a)}")
print(f"Adresa b: {id(seznam_b)}")
print(f"Ukazuji na stejny objekt: {seznam_a is seznam_b}")

seznam_b.append(4)
print(f"Seznam a: {seznam_a}") #Ukazuji ted oba stejne veci i po pridani? Ano

Adresa a: 2142567648576
Adresa b: 2142567648576
Ukazuji na stejny objekt: True
Seznam a: [1, 2, 3, 4]


**2. Garbage Collection a reference counting**

In [1]:
import sys
import gc

text = "Hello World"
print(f"Pocet referenci: {sys. getrefcount(text)}")

text_reference = text
print(f"Nyni pocet referenci: {sys. getrefcount(text)}")

del text_reference
print(f"Nyni pocet referenci: {sys. getrefcount(text)}")

del text
print(f"Nyni pocet referenci: {sys. getrefcount(text)}") #smazana posledni reference, tudiz uz neexistuje

Pocet referenci: 3
Nyni pocet referenci: 4
Nyni pocet referenci: 3


**Pokrocila ukazka**

In [45]:
import time

class Uzel:
    def __init__(self, jmeno):
        self.jmeno = jmeno
        self.dalsi = None

    def __del__(self):
        print(f"Smazan {self.jmeno} z pameti")

uzel_x = Uzel("X")
uzel_y = Uzel("Y")

#Cyklicka reference
uzel_x.dalsi = uzel_y
uzel_y.dalsi = uzel_x

del uzel_x
del uzel_y
print("Po smazani porad zadna zprava z __del__")
time.sleep(2)
print("Spusteni GC")
gc.collect() #Nyni se jiz smazou i z pameti a ukaze se zprava

Po smazani porad zadna zprava z __del__
Spusteni GC
Smazan X z pameti
Smazan Y z pameti


9